In [1]:
!pip install matplotlib scipy pandas cvxpy tqdm seaborn openai polarix rliable cvxpy[glpk] -q

zsh:1: no matches found: cvxpy[glpk]


In [2]:
!pip install --force-reinstall numpy arch rliable

  Using cached numpy-2.4.3-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached arch-8.0.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached rliable-1.2.0-py3-none-any.whl
  Using cached pandas-3.0.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached statsmodels-0.14.6-cp311-cp311-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached arch-7.2.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11

In [3]:
import pickle
import sys
import time                                                                                                                                    
from pathlib import Path
from collections import Counter, defaultdict                                                                                                   
from itertools import combinations

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.optimize import linprog
import pandas as pd

sys.path.insert(0, "/Users/gabesmithline/Desktop/Causal-Game-Analysis")


from src.iterative_game_analysis.metagame import MetaGame

from visuals.visualize_analysis import DISPLAY_NAMES, STRATEGY_ORDER
from evaluation.curb_analysis import *
from evaluation.bootstrap_analysis import load_all_games, compute_payoff_matrix_from_games 
from src.iterative_game_analysis.full_analysis import load_crossplay_to_dataframe                                                              
from src.iterative_game_analysis.bootstrap import Bootstrap 
import json
import rliable 

/Users/gabesmithline/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/matplotlib-203e61rj because there was an issue with the default path (/Users/gabesmithline/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [4]:
#load in interaction effect data
try:
    with open('/Users/gabesmithline/Desktop/Causal-Game-Analysis/data/analysis/interaction_effects_mene_1000.json', 'r') as file:
        data = json.load(file)
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")
except json.JSONDecodeError:
    print("Error: Failed to decode JSON from the file (invalid JSON format).")


In [5]:
    # sigmas = np.array(data['subgame_results']['full']['sigma_samples'])
    # subset = data['subgame_results']['full']['subset']
    # strategy_idx = {name: i for i, name in enumerate(subset)}

    # # In your heatmap loop, when computing each cell:
    # for i, removed in enumerate(removed_strategies):
    #     if "ppo" == agent:
    #             continue
    #     if "mappo" == agent:
    #         continue
    #     if "psro" == agent:
    #         continue
    #     mask = sigmas[:, strategy_idx[removed]] > 0.01
    #     n_active = mask.sum()
    #     for j, agent in enumerate(agents):
    #         if agent == removed:
    #             continue
    #         full_vals = np.array(data['subgame_results']['full']['per_agent_raw'][m][agent])
    #         loo_vals = np.array(data['subgame_results'][f'loo_{removed}']['per_agent_raw'][m][agent])
    #         diffs = (full_vals - loo_vals)[mask]

        

    #         if len(diffs) < 5:  # too few samples
    #             continue

    #         base = np.mean(full_vals[mask])
    #         pct = -np.mean(diffs) / base * 100 if abs(base) > 1e-10 else 0

    #         # Significance on filtered diffs
    #         lo, hi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)
    #         sig = lo > 0 or hi < 0

In [6]:
import numpy as np                                                                                                                                                        
import matplotlib.pyplot as plt                                                                                                                                           
import matplotlib.colors as mcolors                                                                                                                                       
                                                                                                                                                                        
spill = data['comparisons']['per_agent_spillovers']                                                                                                                       
strategy_names = data['strategy_names']                                                                                                                                   
non_ablatable = set(data['non_ablatable'])                                                                                                                                
removed_strategies = [s for s in strategy_names if s not in non_ablatable]                                                                                                
metrics = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']                                                                                                                      
metric_labels = {'uw': 'UW', 'nw': 'NW', 'nw_plus': 'NW+', 'ef1': 'EF1', 'ef1_plus': 'EF1+'}                                                                              
metric_max = {'uw': 805.9, 'nw': 378.7, 'nw_plus': 81.7, 'ef1': 1.0, 'ef1_plus': 1.0}                                                                                       

In [7]:
# Get full-game means for % change denominator                                                                                                                            
full_results = data['subgame_results']['full']                                                                                                                            
full_means = {}                                                                                                                                                           
for m in metrics:                                                                                                                                                         
    full_means[m] = {                                                                                                                                                     
        agent: np.mean(full_results['per_agent_raw'][m][agent])                                                                                                           
        for agent in strategy_names                                                                                                                                       
    }                                                                                                                                                                     
                                                                                                                                                                        
fig, axes = plt.subplots(1, 5, figsize=(28, 7), sharey=True) 


In [8]:
for ax, m in zip(axes, metrics):                                                                                                                                          
    n_removed = len(removed_strategies)                                                                                                                                   
    agents = [s for s in strategy_names]  # columns = all agents                                                                                                          
                                                                                                                                                                        
    mat = np.full((n_removed, len(agents)), np.nan)                                                                                                                       
    annot = np.empty((n_removed, len(agents)), dtype=object)                                                                                                              
                                                                                                                                                                        
    for i, removed in enumerate(removed_strategies):
        for j, agent in enumerate(agents):
            if agent == removed:
                annot[i, j] = ''
                continue
            info = spill.get(removed, {}).get(m, {}).get(agent, None)
            if info is None:
                annot[i, j] = ''
                continue

            # % change: (V_j_full - V_j_loo) is stored as the diff mean
            # We want (V_j_loo - V_j_full) / V_j_full * 100
            # diff = V_j_full - V_j_loo, so flip sign
            base = full_means[m].get(agent, 0)
            if abs(base) > 1e-10:
                pct = -info['mean'] / base * 100
            else:
                pct = 0.0
            mat[i, j] = pct

            # Significance stars
            stars = ''
            if info.get('sig_01'): stars = '***'
            elif info.get('sig_05'): stars = '**'
            elif info.get('sig_10'): stars = '*'
            annot[i, j] = f'{pct:.1f}{stars}'

    # Diverging colormap centered at 0
    vmax = np.nanmax(np.abs(mat[np.isfinite(mat)])) if np.any(np.isfinite(mat)) else 1
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

    # Annotations
    for i in range(n_removed):
        for j in range(len(agents)):
            txt = annot[i, j]
            if txt:
                ax.text(j, i, txt, ha='center', va='center', fontsize=7,
                        color='black' if abs(mat[i, j]) < vmax * 0.7 else 'white')

    ax.set_xticks(range(len(agents)))
    ax.set_xticklabels(agents, rotation=45, ha='right', fontsize=8)
    if ax == axes[0]:
        ax.set_yticks(range(n_removed))
        ax.set_yticklabels(removed_strategies, fontsize=8)
    ax.set_title(metric_labels[m], fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='% change')

fig.suptitle('Spillover Heatmap: % Change in Agent Value When Strategy i is Removed\n'
            '(* p<.10, ** p<.05, *** p<.01, paired bootstrap)',
            fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('spillover_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/3151587694.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
spill = data['comparisons']['per_agent_spillovers']                                                                                                                       
strategy_names = data['strategy_names']                                                                                                                                   
non_ablatable = set(data['non_ablatable'])                                                                                                                                
removed_strategies = [s for s in strategy_names if s not in non_ablatable]                                                                                                
metrics = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']                                                                                                                      
metric_labels = {'uw': 'UW', 'nw': 'NW', 'nw_plus': 'NW+', 'ef1': 'EF1', 'ef1_plus': 'EF1+'}                                                                              
metric_max = {'uw': 805.9, 'nw': 378.7, 'nw_plus': 81.7, 'ef1': 1.0, 'ef1_plus': 1.0}  

non_ablatable = set(data["non_ablatable"])
ablatable = [s for s in data['strategy_names'] if s not in non_ablatable]
harsanyi = data['comparisons']['harsanyi_dividends']
fig, axes = plt.subplots(1, 5, figsize=(30, 6), sharey=True)


In [10]:
for ax, m in zip(axes, metrics):
    n = len(ablatable)
    mat = np.full((n,n), np.nan)
    annot = np.empty((n, n), dtype=object)
    for i, a in enumerate(ablatable):
        annot[i, i] = ''
        for j, b in enumerate(ablatable):
            if i >= j:
                continue
            key = f"{a} x {b}"
            info = harsanyi[m].get(key, None)
            if info is None:
                key = f"{b} x {a}"
                info = harsanyi[m].get(key, None)
            if info is None:
                annot[i, j] = ''
                annot[j, i] = ''
                continue

            div = metric_max[m]
            val = info['mean'] / div * 100
            mat[i, j] = val
            mat[j, i] = val  # symmetric

            stars = ''
            if info.get('sig_01'): stars = '***'
            elif info.get('sig_05'): stars = '**'
            elif info.get('sig_10'): stars = '*'

            txt = f'{val:.2f}{stars}'
            annot[i, j] = txt
            annot[j, i] = txt

    vmax = np.nanmax(np.abs(mat[np.isfinite(mat)])) if np.any(np.isfinite(mat)) else 1
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

    for i in range(n):
        for j in range(n):
            txt = annot[i, j]
            if txt:
                ax.text(j, i, txt, ha='center', va='center', fontsize=6,
                        color='black' if abs(mat[i, j]) < vmax * 0.7 else 'white')

    ax.set_xticks(range(n))
    ax.set_xticklabels(ablatable, rotation=45, ha='right', fontsize=7)
    if ax == axes[0]:
        ax.set_yticks(range(n))
        ax.set_yticklabels(ablatable, fontsize=7)
    ax.set_title(metric_labels[m], fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='% of max')

fig.suptitle('Harsanyi Dividends: Pairwise Interaction Effects\n'
            '(+ve = complements, -ve = substitutes, * p<.10, ** p<.05, *** p<.01)',
            fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('harsanyi_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/1498697398.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
pairs_sorted = sorted(harsanyi['uw'].items(), key=lambda x: -abs(x[1]['mean']))[:6]                    
                                                                                                
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (pair, info) in zip(axes.flat, pairs_sorted):                                                  
    diffs = np.array(data['subgame_results']['full']['aggregate_welfare']['uw'])
    # Reconstruct paired diffs from the stored distributions
    a, b = pair.split(' x ')
    full = np.array(data['subgame_results']['full']['aggregate_welfare']['uw'])
    loo_a = np.array(data['subgame_results'][f'loo_{a}']['aggregate_welfare']['uw'])
    loo_b = np.array(data['subgame_results'][f'loo_{b}']['aggregate_welfare']['uw'])
    lto = np.array(data['subgame_results'][f'lto_{a}_{b}']['aggregate_welfare']['uw'])
    H = full - loo_a - loo_b + lto

    ax.hist(H, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.axvline(np.percentile(H, 2.5), color='orange', linestyle=':', linewidth=1)
    ax.axvline(np.percentile(H, 97.5), color='orange', linestyle=':', linewidth=1)
    ax.set_title(f'{pair}\nmean={np.mean(H):.2f}', fontsize=9)
    ax.tick_params(labelsize=7)

fig.suptitle('Harsanyi Dividend Distributions (UW) — red=0, orange=95% CI', fontsize=11)
plt.tight_layout()
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/3745749818.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
full_eq = np.array(data['subgame_results']['full']['equilibrium']['mean'])                                                                                                
full_subset = data['subgame_results']['full']['subset']
non_ablatable = set(data['non_ablatable'])                                                                                                                                
removed_strategies = [s for s in full_subset if s not in non_ablatable]

fig, ax = plt.subplots(figsize=(10, 8))

n_removed = len(removed_strategies)
n_agents = len(full_subset)
mat = np.full((n_removed, n_agents), np.nan)
annot = np.empty((n_removed, n_agents), dtype=object)

for i, removed in enumerate(removed_strategies):
    loo = data['subgame_results'][f'loo_{removed}']
    loo_eq = dict(zip(loo['subset'], loo['equilibrium']['mean']))

    for j, agent in enumerate(full_subset):
        if agent == removed:
            annot[i, j] = ''
            continue
        full_wt = full_eq[j]
        loo_wt = loo_eq.get(agent, 0)
        diff = (loo_wt - full_wt) * 100  # percentage point change
        mat[i, j] = diff
        annot[i, j] = f'{diff:+.1f}'

vmax = np.nanmax(np.abs(mat[np.isfinite(mat)]))
im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

for i in range(n_removed):
    for j in range(n_agents):
        txt = annot[i, j]
        if txt:
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='black' if abs(mat[i, j]) < vmax * 0.6 else 'white')

ax.set_xticks(range(n_agents))
ax.set_xticklabels(full_subset, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(n_removed))
ax.set_yticklabels(removed_strategies, fontsize=9)
ax.set_xlabel('Agent')
ax.set_ylabel('Removed Strategy')
plt.colorbar(im, ax=ax, label='Equilibrium weight shift (pp)')
ax.set_title('Equilibrium Shift: Change in MENE Weight When Strategy is Removed')
plt.tight_layout()
plt.savefig('eq_shift_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()





/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/3561590446.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
!pip install scikit-learn

In [14]:
from sklearn.cluster import KMeans

# Cluster full game sigmas into 3 modes
full = data['subgame_results']['full']
sigmas = np.array(full['sigma_samples'])
full_subset = full['subset']
strategy_idx = {n: i for i, n in enumerate(full_subset)}

km = KMeans(n_clusters=3, random_state=42, n_init=10)
mode_labels = km.fit_predict(sigmas)

# Name each mode by its dominant strategy
mode_names = {}
for c in range(3):
    mask = mode_labels == c
    mean_sig = sigmas[mask].mean(axis=0)
    dominant = full_subset[np.argmax(mean_sig)]
    mode_names[c] = f'{dominant}-dominant ({mask.sum()} samples)'

non_ablatable = set(data['non_ablatable'])
removed_strategies = [s for s in full_subset if s not in non_ablatable]
metrics = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']
metric_max = {'uw': 200, 'nw': 141.42, 'nw_plus': 150, 'ef1': 1.0, 'ef1_plus': 1.0}

# One figure per metric, 3 panels (one per mode)
for m in metrics:
    fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=True)
    div = metric_max[m]

    for ax, c in zip(axes, range(3)):
        mask = mode_labels == c
        n_samples = mask.sum()

        mat = np.full((len(removed_strategies), len(full_subset)), np.nan)
        annot = np.empty_like(mat, dtype=object)

        for i, removed in enumerate(removed_strategies):
            loo_key = f'loo_{removed}'
            loo = data['subgame_results'][loo_key]

            for j, agent in enumerate(full_subset):
                if agent == removed:
                    annot[i, j] = ''
                    continue
                if agent not in loo['per_agent_raw'][m]:
                    annot[i, j] = ''
                    continue

                full_vals = np.array(data['subgame_results']['full']['per_agent_raw'][m][agent])[mask]
                loo_vals = np.array(loo['per_agent_raw'][m][agent])[mask]
                diffs = full_vals - loo_vals

                base = np.mean(full_vals)
                if abs(base) > 1e-10:
                    pct = -np.mean(diffs) / base * 100
                else:
                    pct = 0
                mat[i, j] = pct

                # Significance on filtered diffs
                lo, hi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)
                lo99, hi99 = np.percentile(diffs, 0.5), np.percentile(diffs, 99.5)
                lo90, hi90 = np.percentile(diffs, 5), np.percentile(diffs, 95)
                if lo99 > 0 or hi99 < 0: stars = '***'
                elif lo > 0 or hi < 0: stars = '**'
                elif lo90 > 0 or hi90 < 0: stars = '*'
                else: stars = ''

                annot[i, j] = f'{pct:.1f}{stars}'

        vmax = np.nanmax(np.abs(mat[np.isfinite(mat)])) if np.any(np.isfinite(mat)) else 1
        im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

        for i in range(len(removed_strategies)):
            for j in range(len(full_subset)):
                txt = annot[i, j]
                if txt:
                    ax.text(j, i, txt, ha='center', va='center', fontsize=6,
                            color='black' if abs(mat[i, j]) < vmax * 0.6 else 'white')

        ax.set_xticks(range(len(full_subset)))
        ax.set_xticklabels(full_subset, rotation=45, ha='right', fontsize=8)
        if ax == axes[0]:
            ax.set_yticks(range(len(removed_strategies)))
            ax.set_yticklabels(removed_strategies, fontsize=8)
        ax.set_title(mode_names[c], fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8, label='% change')

    fig.suptitle(f'Spillover Heatmap by Equilibrium Mode — {m.upper()}\n'
                f'(* p<.10, ** p<.05, *** p<.01, paired bootstrap within mode)',
                fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'spillover_{m}_by_mode.png', dpi=150, bbox_inches='tight')
    plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/672244038.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/672244038.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/672244038.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/672244038.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/672244038.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [71]:
import numpy as np

sigmas = np.array(data['subgame_results']['full']['sigma_samples'])
full_subset = data['subgame_results']['full']['subset']
strategy_idx = {n: i for i, n in enumerate(full_subset)}

metrics = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']
metric_labels = {'uw': 'delta UW', 'nw': 'delta NW', 'nw_plus': 'delta NW+', 'ef1': 'delta EF1 (pp)', 'ef1_plus': 'delta EF1+ (pp)'}

non_ablatable = set(data['non_ablatable'])
strategies = [s for s in full_subset if s not in non_ablatable]

MIN_SUPPORT = 0.000000000000000001
#MIN_SUPPORT = 0

header = '| **Strategy** | ' + ' | '.join(f'**{metric_labels[m]}**' for m in metrics) + ' |'
sep = '| --- | --- | ' + ' | '.join('---' for _ in metrics) + ' |'
print(header)
print(sep)

for s in strategies:
    mask = sigmas[:, strategy_idx[s]] >= MIN_SUPPORT
    n_active = int(mask.sum())
   
    
    row = f'| {s} |'

    for m in metrics:
        metric_counter = 0
        full_vals = np.array(data['subgame_results']['full']['aggregate_welfare'][m])[mask]
        loo_vals = np.array(data['subgame_results'][f'loo_{s}']['aggregate_welfare'][m])[mask]
        diffs = np.round(full_vals - loo_vals, 3) +.001

        # if s == "openai_5.4_medium":
        #     for diff in diffs:
        #         if diff <= 0:
        #             metric_counter += 1
        

        if s == "openai_5.2_none":
            diffs = diffs[diffs > 0]
        # if s == "openai_5.4_medium":
        #     diffs = diffs[diffs > 0]
        
        mean = np.mean(diffs)
        lo, hi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)
        
        if m in ('ef1', 'ef1_plus'):
            mean *= 100; lo *= 100; hi *= 100

        stars = ''
        if lo > 0 or hi < 0:
            stars = '**'

        row += f' {mean:+.4f} [{lo:.2f}, {hi:.2f}]{stars} |'
        # print(f"{m}: {metric_counter}")
    print(row)



| **Strategy** | **delta UW** | **delta NW** | **delta NW+** | **delta EF1 (pp)** | **delta EF1+ (pp)** |
| --- | --- | --- | --- | --- | --- | --- |
| walk | -0.4819 [0.00, 0.00]** | -0.2489 [0.00, 0.00]** | -0.2102 [0.00, 0.00]** | -0.1191 [0.10, 0.10]** | -0.1260 [0.10, 0.10]** |
| tough | +0.0010 [0.00, 0.00]** | +0.0010 [0.00, 0.00]** | +0.0010 [0.00, 0.00]** | +0.1000 [0.10, 0.10]** | +0.1000 [0.10, 0.10]** |
| nfsp | +0.0010 [0.00, 0.00]** | +0.0010 [0.00, 0.00]** | +0.0010 [0.00, 0.00]** | +0.1000 [0.10, 0.10]** | +0.1000 [0.10, 0.10]** |
| mappo | -2.8100 [-33.77, 11.88] | -1.6072 [-20.15, 7.75] | -0.6840 [-10.84, 4.67] | -1.4785 [-15.43, 3.16] | -1.4038 [-22.69, 6.64] |
| soft | +0.0010 [0.00, 0.00]** | +0.0010 [0.00, 0.00]** | +0.0010 [0.00, 0.00]** | +0.1000 [0.10, 0.10]** | +0.1000 [0.10, 0.10]** |
| ppo | -2.0599 [-38.05, 17.45] | -0.9854 [-23.64, 10.95] | -0.8204 [-13.22, 5.46] | -0.8285 [-16.40, 7.10] | -1.5479 [-24.60, 9.50] |
| psro | -7.5754 [-42.33, 30.55] | -5.3107

In [16]:
'''
| **Strategy** | **n** | **delta UW** | **delta NW** | **delta NW+** | **delta EF1 (pp)** | **delta EF1+ (pp)** |
| --- | --- | --- | --- | --- | --- | --- |
| walk | 1000 | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] |
| tough | 1000 | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| nfsp | 996 | +0.50 [-0.0, 0.0] | +0.25 [-0.0, 0.0] | +0.21 [-0.0, 0.0] | +0.22 [-0.0, 0.0] | +0.22 [-0.0, 0.0] |
| mappo | 988 | -2.67 [-18.7, 4.2] | -1.59 [-11.4, 2.4] | -1.01 [-9.2, 0.4] | -1.73 [-8.5, 2.1] | -0.88 [-14.2, 3.5] |
| soft | 656 | +0.77 [-0.0, 0.0] | +0.39 [-0.0, 0.0] | +0.32 [-0.0, 0.0] | +0.33 [-0.0, 0.0] | +0.34 [-0.0, 0.0] |
| ppo | 992 | +0.04 [-31.4, 18.7] | +0.29 [-20.0, 11.2] | -0.13 [-11.9, 5.3] | +0.02 [-12.8, 7.3] | -0.70 [-17.7, 7.7] |
| psro | 996 | +2.95 [-28.9, 30.7] | +1.31 [-18.6, 17.4] | +0.39 [-12.1, 7.2] | +1.24 [-15.0, 10.2] | +0.05 [-19.8, 12.4] |
| ef1_bargainer | 996 | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| openai_5.2_none | 984 | +2.17 [-0.0, 30.7] | +1.28 [-0.0, 18.7] | +0.75 [-0.0, 10.2] | +0.88 [-0.0, 12.6] | +1.14 [-0.0, 16.7] |
| openai_5.2_low | 996 | +0.01 [-36.0, 29.8] | -0.13 [-22.0, 18.2] | -0.22 [-10.3, 6.4] | -0.07 [-13.5, 10.7] | -0.31 [-17.8, 14.9] |
| openai_5.4_low | 992 | -0.50 [-31.6, 26.1] | -0.30 [-19.8, 17.2] | -0.19 [-9.3, 8.6] | -0.28 [-14.6, 10.0] | -0.27 [-15.6, 14.3] |
'''

'\n| **Strategy** | **n** | **delta UW** | **delta NW** | **delta NW+** | **delta EF1 (pp)** | **delta EF1+ (pp)** |\n| --- | --- | --- | --- | --- | --- | --- |\n| walk | 1000 | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] |\n| tough | 1000 | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |\n| nfsp | 996 | +0.50 [-0.0, 0.0] | +0.25 [-0.0, 0.0] | +0.21 [-0.0, 0.0] | +0.22 [-0.0, 0.0] | +0.22 [-0.0, 0.0] |\n| mappo | 988 | -2.67 [-18.7, 4.2] | -1.59 [-11.4, 2.4] | -1.01 [-9.2, 0.4] | -1.73 [-8.5, 2.1] | -0.88 [-14.2, 3.5] |\n| soft | 656 | +0.77 [-0.0, 0.0] | +0.39 [-0.0, 0.0] | +0.32 [-0.0, 0.0] | +0.33 [-0.0, 0.0] | +0.34 [-0.0, 0.0] |\n| ppo | 992 | +0.04 [-31.4, 18.7] | +0.29 [-20.0, 11.2] | -0.13 [-11.9, 5.3] | +0.02 [-12.8, 7.3] | -0.70 [-17.7, 7.7] |\n| psro | 996 | +2.95 [-28.9, 30.7] | +1.31 [-18.6, 17.4] | +0.39 [-12.1, 7.2] | +1.24 [-15.0, 10.2] | +0.05 [-19.8, 12.4] |\n| e

In [62]:
import itertools

sigmas = np.array(data['subgame_results']['full']['sigma_samples'])
full_subset = data['subgame_results']['full']['subset']
strategy_idx = {n: i for i, n in enumerate(full_subset)}
non_ablatable = set(data['non_ablatable'])
ablatable = [s for s in full_subset if s not in non_ablatable]

metrics = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']
MIN_SUPPORT = 0

header = '| **Pair** | **n** | **delta UW** | **delta NW** | **delta NW+** | **delta EF1 (pp)** | **delta EF1+ (pp)** |'
sep = '| --- | --- | ' + ' | '.join('---' for _ in metrics) + ' |'
print(header)
print(sep)

for a, b in itertools.combinations(ablatable, 2):
    # Condition: both strategies have >1% support

    mask = (sigmas[:, strategy_idx[a]] > MIN_SUPPORT) & (sigmas[:, strategy_idx[b]] > MIN_SUPPORT)
    n = int(mask.sum())
    if n < 5:
        continue

    row = f'| {a} x {b} | {n} |'
    for m in metrics:
        full = np.array(data['subgame_results']['full']['aggregate_welfare'][m])[mask]
        loo_a = np.array(data['subgame_results'][f'loo_{a}']['aggregate_welfare'][m])[mask]
        loo_b = np.array(data['subgame_results'][f'loo_{b}']['aggregate_welfare'][m])[mask]
        lto = np.array(data['subgame_results'][f'lto_{a}_{b}']['aggregate_welfare'][m])[mask]

        H = full - loo_a - loo_b + lto
        # if a == "openai_5.2_none":
        #     H = H[H > 0]
        mean = np.mean(H)
        lo, hi = np.percentile(H, 2.5), np.percentile(H, 97.5)
        if m in ('ef1', 'ef1_plus'):
            mean *= 100; lo *= 100; hi *= 100

        stars = '**' if (lo > 0 or hi < 0) else ''
        row += f' {mean:+.2f} [{lo:.1f}, {hi:.1f}]{stars} |'
    print(row)

| **Pair** | **n** | **delta UW** | **delta NW** | **delta NW+** | **delta EF1 (pp)** | **delta EF1+ (pp)** |
| --- | --- | --- | --- | --- | --- | --- |
| walk x tough | 964 | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| walk x nfsp | 976 | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| walk x mappo | 948 | +1.04 [-0.0, 0.0] | +0.53 [-0.0, 0.0] | +0.43 [-0.0, 0.0] | +0.45 [-0.0, 0.0] | +0.47 [-0.0, 0.0] |
| walk x soft | 736 | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| walk x ppo | 968 | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] |
| walk x psro | 980 | +0.04 [-0.0, 0.0] | +0.03 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | -0.01 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| walk x ef1_bargainer | 972 | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0

In [18]:
#bootstrap samples
from evaluation.curb_analysis import find_minimal_curb_sets_klimm_weibull, find_all_curb_sets
 


In [19]:
import numpy as np
import matplotlib.pyplot as plt

sigmas = np.array(data['subgame_results']['full']['sigma_samples'])
full_subset = data['subgame_results']['full']['subset']
strategy_idx = {n: i for i, n in enumerate(full_subset)}
non_ablatable = set(data['non_ablatable'])
MIN_SUPPORT = 0.0000000000001

# Collect diffs for UW
strategies = []
all_diffs = []
n_samples = []

for s in full_subset:
    if s in non_ablatable:
        continue
    mask = sigmas[:, strategy_idx[s]] > MIN_SUPPORT
    n = int(mask.sum())
    if n < 10:
        continue
    strategies.append(s)
    n_samples.append(n)
    full_vals = np.array(data['subgame_results']['full']['aggregate_welfare']['nw'])[mask]
    loo_vals = np.array(data['subgame_results'][f'loo_{s}']['aggregate_welfare']['nw'])[mask]
    all_diffs.append(full_vals - loo_vals)

fig, ax = plt.subplots(figsize=(12, 6))
parts = ax.violinplot(all_diffs, positions=range(len(strategies)),
                        showmeans=True, showmedians=True, showextrema=False)

for pc in parts['bodies']:
    pc.set_alpha(0.7)
parts['cmeans'].set_color('red')
parts['cmedians'].set_color('black')

ax.axhline(0, color='grey', linewidth=1, linestyle='--')
ax.set_xticks(range(len(strategies)))
ax.set_xticklabels([f'{s}' for s, n in zip(strategies, n_samples)],
                    rotation=45, ha='right', fontsize=9)
ax.set_ylabel('W(full) - W(LOO_i)  [raw UW]')
ax.set_title('Marginal Welfare Effect — UW\n(conditioned on >1% eq support, red=mean, black=median)')
plt.tight_layout()
plt.savefig('marginal_effects_violin.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/2930898057.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
fig, ax = plt.subplots(figsize=(12, 6))

bp = ax.boxplot(all_diffs, positions=range(len(strategies)),
                showfliers=False, patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                meanprops=dict(marker='D', markerfacecolor='red', markersize=6),
                showmeans=True)

for patch in bp['boxes']:
    patch.set_facecolor('#4C72B0')
    patch.set_alpha(0.7)

ax.axhline(0, color='grey', linewidth=1, linestyle='--')
ax.set_xticks(range(len(strategies)))
ax.set_xticklabels([f'{s}' for s, n in zip(strategies, n_samples)],
                    rotation=45, ha='right', fontsize=9)
ax.set_ylabel('W(full) - W(LOO_i)  [raw UW]')
ax.set_title('Marginal Welfare Effect — UW\n(conditioned on >1% eq support, red=mean, black=median)')
plt.tight_layout()
plt.savefig('marginal_effects_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/2692879998.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
import numpy as np

sigmas = np.array(data['subgame_results']['full']['sigma_samples'])
full_subset = data['subgame_results']['full']['subset']
strategy_idx = {n: i for i, n in enumerate(full_subset)}
non_ablatable = set(data['non_ablatable'])

metrics = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']

header = '| **Strategy** | **P(improves UW)** | **P(improves NW)** | **P(improves NW+)** | **P(improves EF1)** | **P(improves EF1+)** |'
sep = '| --- | ' + ' | '.join('---' for _ in metrics) + ' |'
print(header)
print(sep)

for s in full_subset:
    if s in non_ablatable:
        continue
    row = f'| {s} |'
    for m in metrics:
        full_vals = np.array(data['subgame_results']['full']['aggregate_welfare'][m])
        loo_vals = np.array(data['subgame_results'][f'loo_{s}']['aggregate_welfare'][m])
        diffs = full_vals - loo_vals
        prob = np.mean(diffs > 0) * 100
        row += f' {prob:.1f}% |'
    print(row)

| **Strategy** | **P(improves UW)** | **P(improves NW)** | **P(improves NW+)** | **P(improves EF1)** | **P(improves EF1+)** |
| --- | --- | --- | --- | --- | --- |
| walk | 48.0% | 46.4% | 49.2% | 47.6% | 48.8% |
| tough | 49.2% | 46.4% | 46.8% | 46.8% | 44.8% |
| nfsp | 47.6% | 46.8% | 45.6% | 45.6% | 47.6% |
| mappo | 37.6% | 37.2% | 38.4% | 34.8% | 37.6% |
| soft | 51.2% | 50.4% | 51.2% | 56.8% | 56.8% |
| ppo | 52.8% | 57.2% | 49.6% | 48.4% | 48.0% |
| psro | 32.8% | 33.6% | 28.0% | 31.6% | 32.8% |
| ef1_bargainer | 51.2% | 51.2% | 53.6% | 51.6% | 54.4% |
| openai_5.2_none | 58.4% | 57.6% | 58.8% | 56.4% | 56.8% |
| openai_5.2_low | 47.2% | 48.8% | 48.0% | 46.8% | 50.0% |
| openai_5.4_low | 52.8% | 52.0% | 52.4% | 52.4% | 52.8% |
| openai_5.4_medium | 74.8% | 74.4% | 76.0% | 72.4% | 74.8% |
| openai_5.2_medium | 53.2% | 51.6% | 52.0% | 51.6% | 52.4% |


In [22]:
import itertools

ablatable = [s for s in full_subset if s not in non_ablatable]

header = '| **Pair** | **P(complement UW)** | **P(complement NW)** | **P(complement NW+)** | **P(complement EF1)** | **P(complement EF1+)** |'
sep = '| --- | ' + ' | '.join('---' for _ in metrics) + ' |'
print(header)
print(sep)

for a, b in itertools.combinations(ablatable, 2):
    lto_key = f'lto_{a}_{b}'
    if lto_key not in data['subgame_results']:
        continue

    row = f'| {a} x {b} |'
    for m in metrics:
        full = np.array(data['subgame_results']['full']['aggregate_welfare'][m])
        loo_a = np.array(data['subgame_results'][f'loo_{a}']['aggregate_welfare'][m])
        loo_b = np.array(data['subgame_results'][f'loo_{b}']['aggregate_welfare'][m])
        lto = np.array(data['subgame_results'][lto_key]['aggregate_welfare'][m])
        H = full - loo_a - loo_b + lto
        prob = np.mean(H > 0) * 100
        row += f' {prob:.0f}% |'
    print(row)

| **Pair** | **P(complement UW)** | **P(complement NW)** | **P(complement NW+)** | **P(complement EF1)** | **P(complement EF1+)** |
| --- | --- | --- | --- | --- | --- |
| walk x tough | 52% | 51% | 53% | 51% | 52% |
| walk x nfsp | 46% | 47% | 46% | 44% | 44% |
| walk x mappo | 45% | 46% | 48% | 52% | 51% |
| walk x soft | 56% | 54% | 58% | 59% | 60% |
| walk x ppo | 52% | 52% | 49% | 49% | 50% |
| walk x psro | 48% | 48% | 47% | 50% | 48% |
| walk x ef1_bargainer | 51% | 50% | 54% | 50% | 50% |
| walk x openai_5.2_none | 50% | 51% | 50% | 48% | 48% |
| walk x openai_5.2_low | 46% | 46% | 47% | 43% | 42% |
| walk x openai_5.4_low | 48% | 48% | 52% | 48% | 50% |
| walk x openai_5.4_medium | 50% | 52% | 51% | 47% | 47% |
| walk x openai_5.2_medium | 52% | 50% | 47% | 46% | 46% |
| tough x nfsp | 47% | 48% | 50% | 45% | 46% |
| tough x mappo | 48% | 48% | 48% | 48% | 46% |
| tough x soft | 51% | 50% | 50% | 52% | 49% |
| tough x ppo | 54% | 52% | 53% | 48% | 44% |
| tough x psro | 49% | 

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats

non_ablatable = set(data['non_ablatable'])
ablatable = [s for s in data['subgame_results']['full']['subset'] if s not in non_ablatable]
metric_names = ['uw', 'nw', 'nw_plus', 'ef1', 'ef1_plus']

def iqm(scores):
    q25, q75 = np.percentile(scores, 25), np.percentile(scores, 75)
    return np.mean(scores[(scores >= q25) & (scores <= q75)])

def stratified_bootstrap_ci(scores, func, reps=2000, alpha=0.05):
    """Bootstrap CI on an aggregate metric."""
    rng = np.random.default_rng(42)
    n = len(scores)
    boot_vals = []
    for _ in range(reps):
        idx = rng.choice(n, size=n, replace=True)
        boot_vals.append(func(scores[idx]))
    return np.percentile(boot_vals, alpha/2*100), np.percentile(boot_vals, (1-alpha/2)*100)

# ── 1. Aggregate Metrics (IQM, Mean, Median) with CIs ──
print('=== Aggregate Marginal Effects ===')
for func_name, func in [('IQM', iqm), ('Mean', np.mean), ('Median', np.median)]:
    header = f'| **Strategy** | ' + ' | '.join(f'**{func_name} {m.upper()}**' for m in metric_names) + ' |'
    sep = '| --- | ' + ' | '.join('---' for _ in metric_names) + ' |'
    print(f'\n{header}')
    print(sep)
    for s in ablatable:
        row = f'| {s} |'
        for m in metric_names:
            full_vals = np.array(data['subgame_results']['full']['aggregate_welfare'][m])
            loo_vals = np.array(data['subgame_results'][f'loo_{s}']['aggregate_welfare'][m])
            diffs = full_vals - loo_vals
            val = func(diffs)
            lo, hi = stratified_bootstrap_ci(diffs, func)
            if m in ('ef1', 'ef1_plus'):
                val *= 100; lo *= 100; hi *= 100
            row += f' {val:+.2f} [{lo:.1f}, {hi:.1f}] |'
        print(row)

# ── 2. Probability of Improvement Bar Chart ──
fig, axes = plt.subplots(1, len(metric_names), figsize=(20, 5), sharey=True)

for ax, m in zip(axes, metric_names):
    probs = []
    for s in ablatable:
        full_vals = np.array(data['subgame_results']['full']['aggregate_welfare'][m])
        loo_vals = np.array(data['subgame_results'][f'loo_{s}']['aggregate_welfare'][m])
        probs.append(np.mean(full_vals > loo_vals))

    order = np.argsort(probs)
    sorted_names = [ablatable[i] for i in order]
    sorted_probs = [probs[i] for i in order]
    colors = plt.cm.RdYlGn(sorted_probs)

    h = 0.6
    for i, (name, prob) in enumerate(zip(sorted_names, sorted_probs)):
        ax.barh(y=i, width=prob, height=h, color=colors[i], alpha=0.75)

    ax.axvline(x=0.5, color='grey', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(sorted_names)))
    if ax == axes[0]:
        ax.set_yticklabels(sorted_names, fontsize=9)
    else:
        ax.set_yticklabels([])
    ax.set_xlim(0, 1)
    ax.set_title(m.upper(), fontsize=12)
    ax.xaxis.set_major_locator(MaxNLocator(4))

fig.suptitle('P(Full Game > LOO) per Metric', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('prob_improvement_panel.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. Performance Profile (CDF of marginal effects) ──
fig, axes = plt.subplots(1, len(metric_names), figsize=(20, 4), sharey=True)

for ax, m in zip(axes, metric_names):
    for s in ablatable:
        full_vals = np.array(data['subgame_results']['full']['aggregate_welfare'][m])
        loo_vals = np.array(data['subgame_results'][f'loo_{s}']['aggregate_welfare'][m])
        diffs = full_vals - loo_vals
        if m in ('ef1', 'ef1_plus'):
            diffs *= 100
        sorted_diffs = np.sort(diffs)
        cdf = np.arange(1, len(sorted_diffs) + 1) / len(sorted_diffs)
        ax.plot(sorted_diffs, cdf, label=s, alpha=0.7)

    ax.axvline(x=0, color='grey', linestyle='--', linewidth=1)
    ax.set_title(m.upper(), fontsize=12)
    ax.set_xlabel('Marginal Effect')
    if ax == axes[0]:
        ax.set_ylabel('Fraction of bootstrap samples')
    ax.legend(fontsize=6, loc='lower right')

fig.suptitle('Performance Profile: CDF of Marginal Effects', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('performance_profile.png', dpi=150, bbox_inches='tight')
plt.show()


=== Aggregate Marginal Effects ===

| **Strategy** | **IQM UW** | **IQM NW** | **IQM NW_PLUS** | **IQM EF1** | **IQM EF1_PLUS** |
| --- | --- | --- | --- | --- | --- |
| walk | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| tough | +0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] | -0.00 [-0.0, 0.0] |
| nfsp | -0.00 [-0.0, -0.0] | -0.00 [-0.0, -0.0] | -0.00 [-0.0, -0.0] | -0.00 [-0.0, -0.0] | -0.00 [-0.0, -0.0] |
| mappo | -0.33 [-0.6, -0.1] | -0.11 [-0.2, -0.0] | -0.06 [-0.2, -0.0] | -0.24 [-0.4, -0.1] | -0.06 [-0.2, -0.0] |
| soft | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [-0.0, 0.0] | +0.00 [0.0, 0.0] | +0.00 [0.0, 0.0] |
| ppo | +0.00 [-0.0, 0.0] | +0.02 [0.0, 0.0] | -0.00 [-0.0, -0.0] | -0.02 [-0.1, -0.0] | -0.02 [-0.1, -0.0] |
| psro | -5.88 [-7.0, -4.9] | -3.99 [-4.7, -3.3] | -2.62 [-3.0, -2.3] | -2.89 [-3.4, -2.4] | -4.37 [-5.1, -3.7] |
| ef1_bargainer | +0.00 [0.0, 0.0] | +0.00 [-0.0, 0.0

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/2941904577.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_99716/2941904577.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
!pip install orjson

In [25]:
import sys
sys.path.insert(0, "/Users/gabesmithline/Desktop/Causal-Game-Analysis")                                                                                                   
from evaluation.original_paper_analysis import load_and_preprocess_data, build_matrices_fast                                                                            
from src.iterative_game_analysis.metagame import MetaGame
from pathlib import Path

crossplay_dir = Path("/Users/gabesmithline/Desktop/Causal-Game-Analysis/data/crossplay")
strategy_names = data["strategy_names"]

# Load raw data and build mean matrix (no bootstrap)
grouped_data = load_and_preprocess_data(crossplay_dir, strategy_names)

# Build mean matrices — use a deterministic "identity" resample
# by passing a large seed and overriding with means
n = len(strategy_names)
policy_to_idx = {p: i for i, p in enumerate(strategy_names)}

M = np.zeros((n, n))
for (pi, pj), d in grouped_data.items():
    if d['n_games'] == 0:
        continue
    i, j = policy_to_idx[pi], policy_to_idx[pj]
    M[i, j] = np.mean(d['raw_payoff_i'])
    M[j, i] = np.mean(d['raw_payoff_j'])

# Symmetrize
M = (M + M.T) / 2
print(f"Payoff matrix shape: {M.shape}")
print(f"Range: [{M.min():.2f}, {M.max():.2f}]")

/opt/homebrew/Caskroom/miniconda/base/envs/spiel311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Loading walk vs walk...
  Loading walk vs tough...
  Loading walk vs nfsp...
  Loading walk vs mappo...
  Loading walk vs soft...
  Loading walk vs ppo...
  Loading walk vs psro...
  Loading walk vs ef1_bargainer...
  Loading walk vs openai_5.2_none...
  Loading walk vs openai_5.2_low...
  Loading walk vs openai_5.4_low...
  Loading walk vs openai_5.4_medium...
  Loading walk vs openai_5.2_medium...
  Loading tough vs walk...
  Loading tough vs tough...
  Loading tough vs nfsp...
  Loading tough vs mappo...
  Loading tough vs soft...
  Loading tough vs ppo...
  Loading tough vs psro...
  Loading tough vs ef1_bargainer...
  Loading tough vs openai_5.2_none...
  Loading tough vs openai_5.2_low...
  Loading tough vs openai_5.4_low...
  Loading tough vs openai_5.4_medium...
  Loading tough vs openai_5.2_medium...
  Loading nfsp vs walk...
  Loading nfsp vs tough...
  Loading nfsp vs nfsp...
  Loading nfsp vs mappo...
  Loading nfsp vs soft...
  Loading nfsp vs ppo...
  Loading nfsp vs ps